In [1]:
import requests
import time
import json
import csv
import re

CTGOV_BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

def fetch_trials(term, start_date="2020-01-01", end_date="MAX"):
    date_filter = f'AREA[StudyFirstPostDate]RANGE[{start_date},{end_date}]'
    funding_filter = 'AREA[LeadSponsorClass]INDUSTRY'
    combined_filter = f'{date_filter} AND {funding_filter}'

    all_studies = []
    page_token = None

    while True:
        params = {
            "query.cond": term,
            "filter.advanced": combined_filter,
            "pageSize": 1000
        }
        if page_token:
            params["pageToken"] = page_token

        r = requests.get(CTGOV_BASE_URL, params=params, timeout=15)
        if r.status_code != 200:
            print(f"Error {r.status_code}, stopping.")
            break

        data = r.json()
        all_studies.extend(data.get("studies", []))
        page_token = data.get("nextPageToken")
        if not page_token:
            break
        time.sleep(0.3)

    return all_studies

def extract_fields(study, seed_term):
    """Pulls the fields needed now AND the ones the diversity scoring will need later."""
    protocol = study.get("protocolSection", {})
    ident = protocol.get("identificationModule", {})
    status_mod = protocol.get("statusModule", {})
    design = protocol.get("designModule", {})
    sponsor_mod = protocol.get("sponsorCollaboratorsModule", {})
    cond_mod = protocol.get("conditionsModule", {})
    conditions_list = cond_mod.get("conditions", [])
    armsMod = protocol.get("armsInterventionsModule", {})
    interventions = armsMod.get("interventions", [])
    drug_names = [
        i.get("name", "") for i in interventions
        if i.get("type") in ("DRUG", "BIOLOGICAL")
    ]

    phases_list = design.get("phases", [])
    if not phases_list:
        phase_value = "BLANK"
    elif phases_list == ["NA"]:
        phase_value = "NA"
    else:
        phase_value = ",".join(phases_list)

    return {
        "nct_id": ident.get("nctId"),
        "title": ident.get("briefTitle"),
        "seed_term": seed_term,
        "study_type": design.get("studyType"),
        "phase": phase_value,
        "status": status_mod.get("overallStatus"),
        "first_posted": status_mod.get("studyFirstPostDateStruct", {}).get("date"),
        "last_update_posted": status_mod.get("lastUpdatePostDateStruct", {}).get("date"),
        "lead_sponsor": sponsor_mod.get("leadSponsor", {}).get("name"),
        "lead_sponsor_class": sponsor_mod.get("leadSponsor", {}).get("class"),
        "conditions": "|".join(conditions_list) if conditions_list else "NA",
        "collaborators": "|".join(
            c.get("name", "") for c in sponsor_mod.get("collaborators", [])
        ),
        "drug_names": "|".join(drug_names) if drug_names else "NA",
        "intervention_types": "|".join(set(i.get("type", "") for i in interventions)) or "NA",
    }

def run_pipeline(seed_term, start_date="2020-01-01", end_date="MAX"):
    print(f"Fetching (MeSH/native query.cond): {seed_term}")

    seen_nct_ids = set()
    rows = []

    studies = fetch_trials(seed_term, start_date=start_date, end_date=end_date)
    for study in studies:
        row = extract_fields(study, seed_term)
        if row["nct_id"] and row["nct_id"] not in seen_nct_ids:
            seen_nct_ids.add(row["nct_id"])
            rows.append(row)

    print(f"  {len(rows)} unique studies found")

    filename = f"{seed_term.replace(' ', '_')}_{start_date}_to_{end_date}_trials.csv"
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"  Saved to {filename}")

    return rows

# --- Run it ---
cardiovascular_trials = run_pipeline("cardiovascular disease", start_date="2020-01-01", end_date="2026-07-01")

Fetching (MeSH/native query.cond): cardiovascular disease
  4257 unique studies found
  Saved to cardiovascular_disease_2020-01-01_to_2026-07-01_trials.csv


# Two Separate Date-Window Pulls

In [2]:
window1 = run_pipeline("cardiovascular disease", start_date="2020-01-01", end_date="2023-12-31")
window2 = run_pipeline("cardiovascular disease", start_date="2023-07-01", end_date="2026-07-01")

Fetching (MeSH/native query.cond): cardiovascular disease
  2583 unique studies found
  Saved to cardiovascular_disease_2020-01-01_to_2023-12-31_trials.csv
Fetching (MeSH/native query.cond): cardiovascular disease
  2021 unique studies found
  Saved to cardiovascular_disease_2023-07-01_to_2026-07-01_trials.csv


# Load both of the window pulls, tag with window, apply remaining filters

In [3]:
import pandas as pd

df1 = pd.read_csv("cardiovascular_disease_2020-01-01_to_2023-12-31_trials.csv", keep_default_na=False)
df1["date_window"] = "2020-01-01 to 2023-12-31"

df2 = pd.read_csv("cardiovascular_disease_2023-07-01_to_2026-07-01_trials.csv", keep_default_na=False)
df2["date_window"] = "2023-07-01 to 2026-07-01"

df_all = pd.concat([df1, df2], ignore_index=True)

# phase != NA (per the agreed filter — industry + date already baked into the fetch)
df_all = df_all[df_all["phase"] != "NA"]

# Join Normalization Table

In [4]:
lookup = pd.read_csv("industry_sponsor_normalization_USABLE_LOOKUP.csv")

df_all["lead_sponsor_normalized"] = df_all["lead_sponsor"].str.strip().str.lower()
lookup["raw_sponsor_name_normalized"] = lookup["raw_sponsor_name"].str.strip().str.lower()

df_all = df_all.merge(
    lookup[["raw_sponsor_name_normalized", "dashboard_sponsor_name"]],
    left_on="lead_sponsor_normalized",
    right_on="raw_sponsor_name_normalized",
    how="left",
    indicator=True
)

df_all["dashboard_sponsor_name"] = df_all["dashboard_sponsor_name"].fillna(df_all["lead_sponsor"])

# Check Phase Values

In [5]:
df_all["phase"].value_counts()

phase
BLANK            1308
PHASE1            515
PHASE3            443
PHASE2            416
PHASE1,PHASE2     165
PHASE4             98
PHASE2,PHASE3      40
EARLY_PHASE1       38
Name: count, dtype: int64

## NEW

In [ ]:
import re

mesh_dict = pd.read_csv("mesh_include_exclude_dictionary.csv")
mesh_dict = mesh_dict[mesh_dict["Disease Area"] == "cardiovascular"]
exclude_terms = mesh_dict[mesh_dict["Section"] == "Exclude"]["Term"].str.lower().tolist()
include_terms_csv = mesh_dict[mesh_dict["Section"] == "Include"]["Term"].str.lower().tolist()


include_keywords = {
    "Coronary/ischemic heart disease": ["coronary", "ischemic heart", "myocardial infarction", "angina"],
    "Heart failure": ["heart failure"],
    "Cardiomyopathy": ["cardiomyopathy"],
    "Arrhythmias": ["arrhythmia", "atrial fibrillation", "atrial flutter",
                     "ventricular tachycardia", "bradycardia", "long qt"],
    "Valvular heart disease": ["valve", "valvular", "mitral", "tricuspid",
                                "aortic stenosis", "aortic regurgitation"],
    "Congenital heart disease": ["congenital heart"],
    "Cerebrovascular/stroke": ["stroke", "cerebrovascular", "transient ischemic attack"],
    "Peripheral arterial disease": ["peripheral arterial", "peripheral vascular"],
    "Aortic disease": ["aortic aneurysm", "aortic dissection"],
    "VTE/PE": ["venous thromboembolism", "deep vein thrombosis", "pulmonary embolism"],
    "Hypertension": ["hypertension", "hypertensive"],
    "CV infections": ["endocarditis", "myocarditis", "pericarditis"],
}
include_pattern = re.compile(
    "|".join(
        [re.escape(t) for t in include_terms_csv]
        + [kw for kws in include_keywords.values() for kw in kws]
    ),
    re.IGNORECASE
)
exclude_pattern = re.compile("|".join(re.escape(t) for t in exclude_terms), re.IGNORECASE)

def classify(conditions_str):
    if not conditions_str or conditions_str == "NA":
        return "needs_review"
    conds = conditions_str.split("|")
    if any(exclude_pattern.search(c) for c in conds):
        return "excluded"
    if any(include_pattern.search(c) for c in conds):
        return "true_cv"
    return "needs_review"

df_all["indication_classification"] = df_all["conditions"].apply(classify)
df_all["is_drug_biologic"] = df_all["intervention_types"].str.contains("DRUG|BIOLOGICAL", na=False)

# Created df_phased

In [7]:
company_early_phases = ["EARLY_PHASE1", "PHASE1", "PHASE1,PHASE2"]
company_late_phases = ["PHASE3", "PHASE4"]

indication_early_phases = ["EARLY_PHASE1", "PHASE1", "PHASE1,PHASE2", "PHASE2"]
indication_late_phases = ["PHASE3", "PHASE4"]

df_phased = df_all[~df_all["phase"].isin(["BLANK", "PHASE2,PHASE3"])]

## NEW

In [8]:
df_observational = df_all[
    (df_all["study_type"] == "OBSERVATIONAL")
    & (df_all["indication_classification"] == "true_cv")
]
df_true_cv_drug = df_phased[
    (df_phased["indication_classification"] == "true_cv")
    & (df_phased["is_drug_biologic"])
]

## Indications

In [9]:
# --- Company views (unchanged from before) ---
df_company_early = df_true_cv_drug[df_true_cv_drug["phase"].isin(company_early_phases)]
df_company_late = df_true_cv_drug[df_true_cv_drug["phase"].isin(company_late_phases)]

top20_companies_all_phases = (
    df_true_cv_drug.groupby("dashboard_sponsor_name")["nct_id"]
    .nunique().sort_values(ascending=False).head(20)
)

top20_companies_early_phase = (
    df_company_early.groupby("dashboard_sponsor_name")["nct_id"]
    .nunique().sort_values(ascending=False).head(20)
)

top20_by_window = (
    df_true_cv_drug.groupby(["date_window", "dashboard_sponsor_name"])["nct_id"]
    .nunique().reset_index()
    .sort_values(["date_window", "nct_id"], ascending=[True, False])
    .groupby("date_window").head(20)
)

# --- Indication views (this is the new/changed part) ---
df_indications = df_true_cv_drug.assign(
    conditions=df_true_cv_drug["conditions"].str.split("|")
).explode("conditions")

top_indications_early = (
    df_indications[df_indications["phase"].isin(indication_early_phases)]
    .groupby("conditions")["nct_id"].nunique()
    .sort_values(ascending=False)
    .head(20)
)

top_indications_late = (
    df_indications[df_indications["phase"].isin(indication_late_phases)]
    .groupby("conditions")["nct_id"].nunique()
    .sort_values(ascending=False)
    .head(20)
)

# --- Export (unchanged) ---
top20_companies_all_phases.to_csv("top20_companies_all_phases.csv")
top20_companies_early_phase.to_csv("top20_companies_early_phase.csv")
top20_by_window.to_csv("top20_companies_by_date_window.csv", index=False)
top_indications_early.to_csv("top_indications_early_phase.csv")
top_indications_late.to_csv("top_indications_late_phase.csv")
df_observational.to_csv("cardiovascular_observational_studies.csv", index=False)

review_list = (
    df_all[df_all["indication_classification"] == "needs_review"]["conditions"]
    .str.split("|").explode().value_counts()
)
review_list.to_csv("conditions_needing_review.csv")

# Prepping For Tableau

In [10]:
df_all.to_csv("cardiovascular_full_dataset_for_tableau.csv", index=False)

# Exploding Collaborators Field

In [11]:
df_collab = df_all.assign(
    collaborators=df_all["collaborators"].str.split("|")
).explode("collaborators")

df_collab = df_collab[df_collab["collaborators"].notna() & (df_collab["collaborators"] != "")]

df_collab.to_csv("cardiovascular_collaborations.csv", index=False)

# Data Validation Check

In [12]:
import pandas as pd
check = pd.read_csv("cardiovascular_collaborations.csv", keep_default_na=False)
pfizer_rows = check[check["dashboard_sponsor_name"] == "Pfizer"]
print(pfizer_rows["collaborators"].tolist())

['The TIMI Study Group', 'Arena is a wholly owned subsidiary of Pfizer', 'Bristol-Myers Squibb', 'Carelon Research', 'National Heart, Lung, and Blood Institute (NHLBI)', 'Bristol-Myers Squibb', 'Bristol-Myers Squibb']
